In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/04.Gold_Helper

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
const_table    = f"{catalog_name}.{silver_schema}.drivers"
nationality_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"

target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
from pyspark.sql.functions import *

In [0]:
drivers_df = spark.read.table(const_table).filter(col("batch_id")==v_batch_id)
nationality_df = spark.read.table(nationality_table)


In [0]:
const_nationality_df= (

    drivers_df.join(nationality_df, on='nationality', how='left')
    .select(
        drivers_df.driver_id,
        drivers_df.driver_name,
        drivers_df.date_of_birth,
        drivers_df.nationality,
        nationality_df.region.alias('nationality_region')
    )
)

In [0]:
columns_to_update=[
        "driver_name",
        "date_of_birth",
        "nationality",
        "nationality_region"
    ]

In [0]:
write_to_gold(input_df=const_nationality_df,target_table=target_table,
                merge_condn="t.driver_id = s.driver_id",
                columns_to_update=columns_to_update
                )

In [0]:
display(spark.table(target_table))